# Tablero descriptivo · Ruta A
Datos BTS/NOAA en HDFS, consulta Spark sobre YARN. Los indicadores son retrospectivos y no representan operaciones LATAM.

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('TableroOperativoRutaA').master('yarn').getOrCreate()
print('Master:', spark.sparkContext.master, 'Spark:', spark.version)
kpi = spark.read.parquet('hdfs:///proyecto_latam/consumo/kpi_aeropuerto_mes')
print('Aeropuertos-mes:', kpi.count())
kpi.orderBy('anio','mes','origin_code').show(15, truncate=False)

In [ ]:
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
tabla = (kpi.groupBy('origin_code')
         .agg(F.sum('vuelos').alias('vuelos'), F.sum('vuelos_demorados_15').alias('demorados'),
              (F.sum(F.col('cobertura_clima') * F.col('vuelos')) / F.sum('vuelos')).alias('cobertura'))
         .withColumn('tasa', F.col('demorados') / F.col('vuelos'))
         .orderBy('origin_code').toPandas())
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, campo, titulo in zip(axes.flat, ['vuelos','demorados','tasa','cobertura'],
                             ['Vuelos', 'Vuelos demorados ≥15 min', 'Tasa de demora', 'Cobertura NOAA']):
    ax.bar(tabla['origin_code'], tabla[campo])
    ax.set_title(titulo)
    ax.tick_params(axis='x', rotation=45)
fig.tight_layout()
plt.show()

**Interpretación:** comparar aeropuertos con tasas y volumen; las cifras concretas deben redactarse después de ejecutar esta notebook. Responsable propuesto: analista de operaciones. Frecuencia: mensual tras actualizar las fuentes.

In [ ]:
spark.stop()